In [4]:
import numpy as np
import pandas as pd

# =========================================================
# GEOMETRY DESCRIPTOR EXTRACTION PROGRAM
# =========================================================
#
# Extracts:
#
# 1. M-M Distance
# 2. Shortest Intramolecular M-N Distance
# 3. Monomer Centroid Slippage
#
# Slippage =
# perpendicular displacement between monomer centroids
# relative to M-M interaction axis
#
# Monomers assigned geometrically
# using nearest-metal criterion
#
# Heavy atoms only used for centroids
#
# =========================================================

# ---------------------------------------------------------
# INPUT FILE
# ---------------------------------------------------------

coord_file = "All_Cu_Coordinates.txt"

# ---------------------------------------------------------
# READ FILE
# ---------------------------------------------------------

with open(coord_file, "r") as f:
    lines = f.readlines()

# ---------------------------------------------------------
# STORE COMPLEXES
# ---------------------------------------------------------

complexes = {}
current_complex = None

for line in lines:

    stripped = line.strip()

    # Detect complex header
    if stripped.startswith("Complex"):

        current_complex = stripped.split(":")[1].strip()
        complexes[current_complex] = []

    # Skip separators and blank lines
    elif stripped.startswith("=") or stripped == "":
        continue

    else:

        parts = stripped.split()

        if len(parts) == 4:

            atom = parts[0]

            x = float(parts[1])
            y = float(parts[2])
            z = float(parts[3])

            coord = np.array([x, y, z])

            complexes[current_complex].append(
                (atom, coord)
            )

# ---------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------

def distance(a, b):

    return np.linalg.norm(a - b)

# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------

results = []

# ---------------------------------------------------------
# PROCESS EACH COMPLEX
# ---------------------------------------------------------

for complex_name, atoms in complexes.items():

    # -----------------------------------------------------
    # IDENTIFY METALS
    # -----------------------------------------------------

    metal_atoms = []

    for atom, coord in atoms:

        if atom in ["Cu", "Ag", "Au"]:

            metal_atoms.append((atom, coord))

    # Must contain exactly two metals
    if len(metal_atoms) != 2:

        print(f"Skipping {complex_name} : metal issue")
        continue

    metal_symbol = metal_atoms[0][0]

    M1 = metal_atoms[0][1]
    M2 = metal_atoms[1][1]

    # -----------------------------------------------------
    # M-M DISTANCE
    # -----------------------------------------------------

    MM_distance = distance(M1, M2)

    # -----------------------------------------------------
    # GEOMETRIC MONOMER ASSIGNMENT
    # -----------------------------------------------------

    monomer1 = []
    monomer2 = []

    for atom, coord in atoms:

        d1 = distance(coord, M1)
        d2 = distance(coord, M2)

        if d1 <= d2:

            monomer1.append((atom, coord))

        else:

            monomer2.append((atom, coord))

    # -----------------------------------------------------
    # SHORTEST INTRAMOLECULAR M-N
    # -----------------------------------------------------

    N1_distances = []
    N2_distances = []

    for atom, coord in monomer1:

        if atom == "N":

            N1_distances.append(
                distance(M1, coord)
            )

    for atom, coord in monomer2:

        if atom == "N":

            N2_distances.append(
                distance(M2, coord)
            )

    # Skip problematic systems
    if len(N1_distances) == 0 or len(N2_distances) == 0:

        print(f"Skipping {complex_name} : N assignment issue")
        continue

    shortest_M1_N = min(N1_distances)
    shortest_M2_N = min(N2_distances)

    shortest_MN = (
        shortest_M1_N + shortest_M2_N
    ) / 2

    # -----------------------------------------------------
    # HEAVY-ATOM CENTROIDS
    # -----------------------------------------------------

    heavy_atoms_1 = []
    heavy_atoms_2 = []

    for atom, coord in monomer1:

        if atom != "H":

            heavy_atoms_1.append(coord)

    for atom, coord in monomer2:

        if atom != "H":

            heavy_atoms_2.append(coord)

    centroid1 = np.mean(
        heavy_atoms_1,
        axis=0
    )

    centroid2 = np.mean(
        heavy_atoms_2,
        axis=0
    )

    # -----------------------------------------------------
    # MONOMER SLIPPAGE
    # -----------------------------------------------------

    # M-M interaction axis

    MM_axis = M2 - M1

    MM_unit = (
        MM_axis / np.linalg.norm(MM_axis)
    )

    # Centroid displacement

    centroid_vector = (
        centroid2 - centroid1
    )

    # Projection along M-M axis

    projection_length = np.dot(
        centroid_vector,
        MM_unit
    )

    projection_vector = (
        projection_length * MM_unit
    )

    # Perpendicular component

    perpendicular_vector = (
        centroid_vector - projection_vector
    )

    slippage = np.linalg.norm(
        perpendicular_vector
    )

    # -----------------------------------------------------
    # STORE RESULTS
    # -----------------------------------------------------

    results.append({

        "Complex": complex_name,

        "Metal": metal_symbol,

        "M_M_Distance": round(
            MM_distance, 4
        ),

        "Shortest_M_N": round(
            shortest_MN, 4
        ),

        "Monomer_Slippage": round(
            slippage, 4
        )

    })

# ---------------------------------------------------------
# CREATE DATAFRAME
# ---------------------------------------------------------

df = pd.DataFrame(results)

# ---------------------------------------------------------
# DISPLAY SETTINGS
# ---------------------------------------------------------

pd.set_option(
    'display.max_rows',
    None
)

pd.set_option(
    'display.max_columns',
    None
)

# ---------------------------------------------------------
# DISPLAY RESULTS
# ---------------------------------------------------------

print(df)

# ---------------------------------------------------------
# OPTIONAL SAVE
# ---------------------------------------------------------

df.to_csv(
     "Geometry_Descriptors.csv",
     index=False
)

    Complex Metal  M_M_Distance  Shortest_M_N  Monomer_Slippage
0      Ag_1    Ag        3.0375        2.0693            1.0665
1      Ag_2    Ag        3.0125        2.0885            2.1310
2      Ag_3    Ag        3.0250        2.0650            4.0887
3      Ag_4    Ag        3.0125        2.0594            2.1595
4      Ag_5    Ag        3.0375        2.0685            2.1696
5      Ag_6    Ag        3.0375        2.0690            2.1351
6      Ag_7    Ag        3.0250        2.0795            1.0745
7      Ag_8    Ag        3.0125        2.0866            1.0339
8      Ag_9    Ag        2.9875        2.1056            2.1098
9     Ag_10    Ag        3.0000        2.0829            4.0853
10    Ag_11    Ag        3.0000        2.0768            2.1386
11    Ag_12    Ag        3.0125        2.0834            2.1466
12    Ag_13    Ag        3.0250        2.0833            2.1115
13    Ag_14    Ag        3.0125        2.0945            1.0405
14    Ag_15    Ag        3.0000        2

In [1]:
import numpy as np
import pandas as pd

# =========================================================
# GEOMETRY + HX_NEAREST DESCRIPTOR EXTRACTION
# =========================================================
#
# Extracts:
#
# 1. M-M Distance
# 2. Shortest Intramolecular M-N Distance
# 3. Monomer Slippage
# 4. HX_nearest
#
# HX_nearest =
# shortest H---X distance in Py systems
#
# CN systems:
# HX_nearest = NaN
#
# =========================================================

coord_file = "All_Cu_Coordinates.txt"

# ---------------------------------------------------------
# READ FILE
# ---------------------------------------------------------

with open(coord_file, "r") as f:
    lines = f.readlines()

# ---------------------------------------------------------
# STORE COMPLEXES
# ---------------------------------------------------------

complexes = {}
current_complex = None

for line in lines:

    stripped = line.strip()

    if stripped.startswith("Complex"):

        current_complex = stripped.split(":")[1].strip()
        complexes[current_complex] = []

    elif stripped.startswith("=") or stripped == "":
        continue

    else:

        parts = stripped.split()

        if len(parts) == 4:

            atom = parts[0]

            x = float(parts[1])
            y = float(parts[2])
            z = float(parts[3])

            coord = np.array([x, y, z])

            complexes[current_complex].append(
                (atom, coord)
            )

# ---------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------

def distance(a, b):
    return np.linalg.norm(a - b)

# ---------------------------------------------------------
# LIGAND IDENTIFICATION
# ---------------------------------------------------------

def identify_ligand(complex_name):

    metal, num = complex_name.split("_")
    num = int(num)

    if metal == "Ag":

        if num <= 28:
            return "CN"
        else:
            return "Py"

    elif metal == "Au":

        if num <= 70:
            return "CN"
        else:
            return "Py"

    elif metal == "Cu":

        if num <= 25:
            return "CN"
        else:
            return "Py"

    return None

# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------

results = []

# ---------------------------------------------------------
# PROCESS EACH COMPLEX
# ---------------------------------------------------------

for complex_name, atoms in complexes.items():

    ligand_type = identify_ligand(complex_name)

    # -----------------------------------------------------
    # IDENTIFY METALS
    # -----------------------------------------------------

    metal_atoms = []

    for atom, coord in atoms:

        if atom in ["Cu", "Ag", "Au"]:

            metal_atoms.append(
                (atom, coord)
            )

    if len(metal_atoms) != 2:

        print(
            f"Skipping {complex_name} : metal issue"
        )
        continue

    metal_symbol = metal_atoms[0][0]

    M1 = metal_atoms[0][1]
    M2 = metal_atoms[1][1]

    # -----------------------------------------------------
    # M-M DISTANCE
    # -----------------------------------------------------

    MM_distance = distance(M1, M2)

    # -----------------------------------------------------
    # GEOMETRIC MONOMER ASSIGNMENT
    # -----------------------------------------------------

    monomer1 = []
    monomer2 = []

    for atom, coord in atoms:

        d1 = distance(coord, M1)
        d2 = distance(coord, M2)

        if d1 <= d2:

            monomer1.append(
                (atom, coord)
            )

        else:

            monomer2.append(
                (atom, coord)
            )

    # -----------------------------------------------------
    # SHORTEST M-N
    # -----------------------------------------------------

    N1_distances = []
    N2_distances = []

    for atom, coord in monomer1:

        if atom == "N":

            N1_distances.append(
                distance(M1, coord)
            )

    for atom, coord in monomer2:

        if atom == "N":

            N2_distances.append(
                distance(M2, coord)
            )

    if len(N1_distances) == 0 or len(N2_distances) == 0:

        print(
            f"Skipping {complex_name} : N issue"
        )
        continue

    shortest_MN = (
        min(N1_distances)
        + min(N2_distances)
    ) / 2

    # -----------------------------------------------------
    # HEAVY ATOM CENTROIDS
    # -----------------------------------------------------

    heavy_atoms_1 = [
        coord
        for atom, coord in monomer1
        if atom != "H"
    ]

    heavy_atoms_2 = [
        coord
        for atom, coord in monomer2
        if atom != "H"
    ]

    centroid1 = np.mean(
        heavy_atoms_1,
        axis=0
    )

    centroid2 = np.mean(
        heavy_atoms_2,
        axis=0
    )

    # -----------------------------------------------------
    # MONOMER SLIPPAGE
    # -----------------------------------------------------

    MM_axis = M2 - M1

    MM_unit = (
        MM_axis /
        np.linalg.norm(MM_axis)
    )

    centroid_vector = (
        centroid2 - centroid1
    )

    projection_length = np.dot(
        centroid_vector,
        MM_unit
    )

    projection_vector = (
        projection_length *
        MM_unit
    )

    perpendicular_vector = (
        centroid_vector -
        projection_vector
    )

    slippage = np.linalg.norm(
        perpendicular_vector
    )

    # -----------------------------------------------------
    # HX_NEAREST
    # -----------------------------------------------------

    HX_nearest = np.nan

    if ligand_type == "Py":

        H_atoms = []
        X_atoms = []

        for atom, coord in atoms:

            if atom == "H":
                H_atoms.append(coord)

            if atom in [
                "F",
                "Cl",
                "Br",
                "I"
            ]:
                X_atoms.append(coord)

        if (
            len(H_atoms) > 0
            and
            len(X_atoms) > 0
        ):

            all_distances = []

            for H in H_atoms:

                for X in X_atoms:

                    all_distances.append(
                        distance(H, X)
                    )

            HX_nearest = min(
                all_distances
            )

    # -----------------------------------------------------
    # STORE RESULTS
    # -----------------------------------------------------

    results.append({

        "Complex":
            complex_name,

        "Metal":
            metal_symbol,

        "Ligand":
            ligand_type,

        "M_M_Distance":
            round(
                MM_distance,
                4
            ),

        "Shortest_M_N":
            round(
                shortest_MN,
                4
            ),

        "Monomer_Slippage":
            round(
                slippage,
                4
            ),

        "HX_nearest":
            round(
                HX_nearest,
                4
            )
            if not pd.isna(
                HX_nearest
            )
            else np.nan

    })

# ---------------------------------------------------------
# DATAFRAME
# ---------------------------------------------------------

df = pd.DataFrame(results)

pd.set_option(
    "display.max_rows",
    None
)

pd.set_option(
    "display.max_columns",
    None
)

print(df)

# ---------------------------------------------------------
# HX SUMMARY
# ---------------------------------------------------------

print("\n")
print("=" * 60)
print("HX_nearest Summary")
print("=" * 60)

print(
    df["HX_nearest"]
    .describe()
)

# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

df.to_csv(
    "Geometry_Descriptors_with_HX.csv",
    index=False
)

print("\nSaved:")
print(
    "Geometry_Descriptors_with_HX.csv"
)

    Complex Metal Ligand  M_M_Distance  Shortest_M_N  Monomer_Slippage  \
0      Ag_1    Ag     CN        3.0375        2.0693            1.0665   
1      Ag_2    Ag     CN        3.0125        2.0885            2.1310   
2      Ag_3    Ag     CN        3.0250        2.0650            4.0887   
3      Ag_4    Ag     CN        3.0125        2.0594            2.1595   
4      Ag_5    Ag     CN        3.0375        2.0685            2.1696   
5      Ag_6    Ag     CN        3.0375        2.0690            2.1351   
6      Ag_7    Ag     CN        3.0250        2.0795            1.0745   
7      Ag_8    Ag     CN        3.0125        2.0866            1.0339   
8      Ag_9    Ag     CN        2.9875        2.1056            2.1098   
9     Ag_10    Ag     CN        3.0000        2.0829            4.0853   
10    Ag_11    Ag     CN        3.0000        2.0768            2.1386   
11    Ag_12    Ag     CN        3.0125        2.0834            2.1466   
12    Ag_13    Ag     CN        3.0250

In [2]:
df[
    ["Complex",
     "Ligand",
     "HX_nearest"]
].head(20)

,Complex,Ligand,HX_nearest
0,Ag_1,CN,NaN
1,Ag_2,CN,NaN
2,Ag_3,CN,NaN
3,Ag_4,CN,NaN
4,Ag_5,CN,NaN
5,Ag_6,CN,NaN
6,Ag_7,CN,NaN
7,Ag_8,CN,NaN
8,Ag_9,CN,NaN
9,Ag_10,CN,NaN


In [3]:
df["HX_nearest"].describe()

count    49.000000
mean      2.813786
std       0.526411
min       2.168200
25%       2.502000
50%       2.603100
75%       2.900600
max       3.753700
Name: HX_nearest, dtype: float64

In [4]:
df.sort_values(
    "HX_nearest"
)[
    ["Complex",
     "Ligand",
     "HX_nearest"]
].head(20)

,Complex,Ligand,HX_nearest
112,Cu_29,Py,2.1682
111,Cu_28,Py,2.1751
110,Cu_27,Py,2.1774
114,Cu_31,Py,2.1884
109,Cu_26,Py,2.2013
113,Cu_30,Py,2.2112
115,Cu_32,Py,2.2624
139,Cu_56,Py,2.4587
132,Cu_49,Py,2.4736
118,Cu_35,Py,2.4787


In [5]:
df.sort_values(
    "HX_nearest",
    ascending=False
)[
    ["Complex",
     "Ligand",
     "HX_nearest"]
].head(20)

,Complex,Ligand,HX_nearest
73,Au_74,Py,3.7537
32,Ag_33,Py,3.7471
75,Au_76,Py,3.7442
74,Au_75,Py,3.7373
77,Au_78,Py,3.7282
76,Au_77,Py,3.7282
70,Au_71,Py,3.7238
33,Ag_34,Py,3.7056
31,Ag_32,Py,3.6920
34,Ag_35,Py,3.6853


In [7]:
import pandas as pd

# Geometry descriptor file
geom_df = pd.read_csv(
    "Geometry_Descriptors.csv"
)

# Original dataset containing E
data_df = pd.read_csv(
    "updated_metallophilic_dataset.csv"
)

# Check common identifier column
print(geom_df.columns)
print(data_df.columns)

Index(['Complex', 'Metal', 'Ligand', 'M_M_Distance', 'Shortest_M_N',
       'Monomer_Slippage', 'HX_nearest'],
      dtype='object')
Index(['dimer', 'Metal', 'Ligand', 'X', 'R', 'Y_Smiles', 'E', 'd', 'rho',
       'laplacian', 'V', 'G', 'V_over_G'],
      dtype='object')


In [8]:
merged_df = geom_df.merge(
    data_df[['dimer', 'E']],
    left_on='Complex',
    right_on='dimer',
    how='left'
)

print(merged_df.head())

  Complex Metal Ligand  M_M_Distance  Shortest_M_N  Monomer_Slippage  \
0    Ag_1    Ag     CN        3.0375        2.0693            1.0665   
1    Ag_2    Ag     CN        3.0125        2.0885            2.1310   
2    Ag_3    Ag     CN        3.0250        2.0650            4.0887   
3    Ag_4    Ag     CN        3.0125        2.0594            2.1595   
4    Ag_5    Ag     CN        3.0375        2.0685            2.1696   

   HX_nearest dimer     E  
0         NaN  Ag_1 -6.16  
1         NaN  Ag_2 -6.57  
2         NaN  Ag_3 -6.67  
3         NaN  Ag_4 -6.58  
4         NaN  Ag_5 -6.00  


In [9]:
cols = [
    'HX_nearest',
    'M_M_Distance',
    'Shortest_M_N',
    'Monomer_Slippage',
    'E'
]

print(
    merged_df[cols].corr()
)

                  HX_nearest  M_M_Distance  Shortest_M_N  Monomer_Slippage  \
HX_nearest          1.000000      0.555368      0.613210         -0.304950   
M_M_Distance        0.555368      1.000000      0.698573         -0.382420   
Shortest_M_N        0.613210      0.698573      1.000000         -0.108725   
Monomer_Slippage   -0.304950     -0.382420     -0.108725          1.000000   
E                   0.572802      0.748988      0.348180         -0.625675   

                         E  
HX_nearest        0.572802  
M_M_Distance      0.748988  
Shortest_M_N      0.348180  
Monomer_Slippage -0.625675  
E                 1.000000  


In [10]:
py_df = merged_df[
    merged_df['Ligand'] == 'Py'
]

print(
    py_df[cols].corr()
)

                  HX_nearest  M_M_Distance  Shortest_M_N  Monomer_Slippage  \
HX_nearest          1.000000      0.555368      0.613210         -0.304950   
M_M_Distance        0.555368      1.000000      0.821304         -0.183980   
Shortest_M_N        0.613210      0.821304      1.000000         -0.190081   
Monomer_Slippage   -0.304950     -0.183980     -0.190081          1.000000   
E                   0.572802      0.933732      0.811154         -0.277644   

                         E  
HX_nearest        0.572802  
M_M_Distance      0.933732  
Shortest_M_N      0.811154  
Monomer_Slippage -0.277644  
E                 1.000000  
